# Predicción de Partidos — Mundial FIFA 2026

## Introducción

Este notebook presenta un análisis completo del proyecto **World Cup 2026**, que combina:
- **Análisis exploratorio** de datos históricos de Mundiales FIFA (2014, 2018, 2022)
- **Ingeniería de features** para entrenar un modelo de Machine Learning
- **Predicción de resultados** de partidos usando Random Forest
- **Visualizaciones interactivas** con Plotly

### Fuentes de datos

| Base de datos | Contenido |
|---|---|
| `historical.db` | Partidos de eliminatorias de los Mundiales 2014, 2018 y 2022 |
| `worldcup.db` | Datos del Mundial 2026: 48 selecciones, estadios, partidos programados |

### Metodología

El modelo de predicción utiliza un **Random Forest Classifier** con 10 features derivados del historial de cada selección: tasa de victorias, promedio de goles a favor/en contra, confederación, diferencia de ranking FIFA, y si el partido es de eliminatorias.

---

**Autor:** Álvaro  
**Fecha:** Septiembre 2026  
**Repositorio:** [github-limpio/worldcup-2026](https://github.com/username/worldcup-2026)

---
## 1. Carga de Datos

In [ ]:
import sqlite3
import warnings
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings("ignore")

BASE = Path().resolve().parent
HIST_DB = BASE / "data" / "historical.db"
WC_DB = BASE / "data" / "worldcup.db"

print(f"Proyecto base: {BASE}")
print(f"DB histórica: {HIST_DB} ({HIST_DB.exists()})")
print(f"DB 2026: {WC_DB} ({WC_DB.exists()})")

In [ ]:
def query(sql, db_path):
    conn = sqlite3.connect(str(db_path))
    df = pd.read_sql_query(sql, conn)
    conn.close()
    return df


# Datos históricos
matches_hist = query("SELECT * FROM matches", HIST_DB)
teams_hist = query("SELECT * FROM teams", HIST_DB)
world_cups = query("SELECT * FROM world_cups ORDER BY year", HIST_DB)

# Datos Mundial 2026
teams_2026 = query("SELECT * FROM teams ORDER BY fifa_ranking", WC_DB)
matches_2026 = query(
    "SELECT m.*, t1.name as home, t2.name as away "
    "FROM matches m "
    "JOIN teams t1 ON m.home_team_id = t1.team_id "
    "JOIN teams t2 ON m.away_team_id = t2.team_id",
    WC_DB,
)

print(f"Partidos históricos: {len(matches_hist)}")
print(f"Equipos históricos: {teams_hist['name'].nunique()}")
print(f"Ediciones WC: {world_cups['year'].tolist()}")
print(f"Equipos WC 2026: {len(teams_2026)}")
print(f"Partidos WC 2026: {len(matches_2026)}")

In [ ]:
print("=== Estructura de partidos históricos ===")
matches_hist.head(10)

In [ ]:
print("=== Selecciones WC 2026 (Top 10 por ranking) ===")
teams_2026[["name", "fifa_ranking", "confederation"]].head(10)

---
## 2. Análisis Exploratorio

In [ ]:
print("=== Resumen por edición del Mundial ===")
print(
    world_cups[
        ["year", "host", "champion", "runner_up", "total_goals", "total_matches"]
    ].to_string(index=False)
)

In [ ]:
# Evolución de goles por edición
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=["Goles totales por edición", "Promedio de goles por partido"],
)

fig.add_trace(
    go.Bar(
        x=world_cups["year"],
        y=world_cups["total_goals"],
        text=world_cups["total_goals"],
        textposition="outside",
        marker_color=["#636EFA", "#EF553B", "#00CC96"],
        name="Goles",
    ),
    row=1,
    col=1,
)

avg_goals = (world_cups["total_goals"] / world_cups["total_matches"]).round(2)
fig.add_trace(
    go.Scatter(
        x=world_cups["year"],
        y=avg_goals,
        mode="lines+markers+text",
        text=[str(v) for v in avg_goals],
        textposition="top center",
        line=dict(color="#AB63FA", width=3),
        name="Promedio",
    ),
    row=1,
    col=2,
)

fig.update_layout(height=400, showlegend=False, template="plotly_white")
fig.show()

In [ ]:
# Distribución de resultados en partidos históricos
matches_hist["result"] = matches_hist.apply(
    lambda r: (
        "Victoria local"
        if r["home_score"] > r["away_score"]
        else ("Victoria visitante" if r["away_score"] > r["home_score"] else "Empate")
    ),
    axis=1,
)

result_counts = matches_hist["result"].value_counts()

fig = px.pie(
    values=result_counts.values,
    names=result_counts.index,
    title="Distribución de resultados en eliminatorias (2014-2022)",
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig.update_layout(template="plotly_white")
fig.show()

print(result_counts)

In [ ]:
# Top selecciones con más victorias en Mundiales
def compute_team_record(df):
    records = []
    for _, r in df.iterrows():
        home, away = r["home_team"], r["away_team"]
        hs, aws = r["home_score"], r["away_score"]
        records.append(
            {
                "team": home,
                "gf": hs,
                "ga": aws,
                "result": "W" if hs > aws else ("D" if hs == aws else "L"),
            }
        )
        records.append(
            {
                "team": away,
                "gf": aws,
                "ga": hs,
                "result": "W" if aws > hs else ("D" if hs == aws else "L"),
            }
        )
    return pd.DataFrame(records)


team_records = compute_team_record(matches_hist)
team_summary = (
    team_records.groupby("team")
    .agg(
        matches=("result", "count"),
        wins=("result", lambda x: (x == "W").sum()),
        gf=("gf", "sum"),
        ga=("ga", "sum"),
    )
    .reset_index()
)
team_summary["win_rate"] = (team_summary["wins"] / team_summary["matches"] * 100).round(
    1
)
team_summary["gd"] = team_summary["gf"] - team_summary["ga"]
team_summary = team_summary.sort_values("wins", ascending=False)

fig = px.bar(
    team_summary.head(15),
    x="team",
    y="wins",
    title="Top 15 selecciones con más victorias en Mundiales (2014-2022)",
    text="win_rate",
    color="wins",
    color_continuous_scale="Viridis",
)
fig.update_traces(texttemplate="%{text}%", textposition="outside")
fig.update_layout(
    xaxis_title="Selección",
    yaxis_title="Victorias",
    template="plotly_white",
    height=450,
)
fig.show()

In [ ]:
# Promedio de goles por ronda
round_stats = (
    matches_hist.groupby("round")
    .agg(
        matches=("id", "count"),
        total_goals=(
            "home_score",
            lambda x: x.sum() + matches_hist.loc[x.index, "away_score"].sum(),
        ),
    )
    .reset_index()
)
round_order = ["Round of 16", "Quarterfinal", "Semifinal", "Third Place", "Final"]
round_stats["round"] = pd.Categorical(
    round_stats["round"], categories=round_order, ordered=True
)
round_stats = round_stats.sort_values("round")
round_stats["avg_goals"] = (round_stats["total_goals"] / round_stats["matches"]).round(
    2
)

fig = make_subplots(
    rows=1, cols=2, subplot_titles=["Goles totales por ronda", "Promedio por partido"]
)

fig.add_trace(
    go.Bar(
        x=round_stats["round"],
        y=round_stats["total_goals"],
        text=round_stats["total_goals"],
        textposition="outside",
        marker_color="#EF553B",
        name="Total",
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Bar(
        x=round_stats["round"],
        y=round_stats["avg_goals"],
        text=round_stats["avg_goals"],
        textposition="outside",
        marker_color="#636EFA",
        name="Promedio",
    ),
    row=1,
    col=2,
)

fig.update_layout(height=400, showlegend=False, template="plotly_white")
fig.show()

In [ ]:
# Distribución por confederación
conf_counts = teams_2026["confederation"].value_counts().reset_index()
conf_counts.columns = ["confederation", "count"]

fig = px.pie(
    conf_counts,
    values="count",
    names="confederation",
    title="Distribución de selecciones por confederación — Mundial 2026",
    hole=0.3,
    color_discrete_sequence=px.colors.qualitative.Pastel,
)
fig.update_traces(textposition="inside", textinfo="percent+label")
fig.update_layout(template="plotly_white")
fig.show()

---
## 3. Ingeniería de Features

Para entrenar el modelo de predicción, transformamos cada partido en un vector de 10 features:

| Feature | Descripción |
|---|---|
| `is_knockout` | 1 si es eliminatoria, 0 si fase de grupos |
| `home_win_rate` | Tasa de victorias del local acumulada |
| `away_win_rate` | Tasa de victorias del visitante acumulada |
| `home_avg_gf` | Promedio de goles a favor del local |
| `away_avg_gf` | Promedio de goles a favor del visitante |
| `home_avg_ga` | Promedio de goles en contra del local |
| `away_avg_ga` | Promedio de goles en contra del visitante |
| `home_conf_enc` | Confederación del local (codificada) |
| `away_conf_enc` | Confederación del visitante (codificada) |
| `rank_diff` | Diferencia de ranking FIFA (local - visitante) |

> **Nota:** Los stats se calculan de forma acumulativa (rolling), es decir, cada partido solo "ve" el historial previo de cada selección, simulando un escenario real de predicción.

In [ ]:
# Preparar datos de entrenamiento (misma lógica que match_predictor.py)
matches = matches_hist.rename(columns={"home_team": "home", "away_team": "away"})
matches["label"] = matches.apply(
    lambda r: (
        "home_win"
        if r["home_score"] > r["away_score"]
        else ("away_win" if r["away_score"] > r["home_score"] else "draw")
    ),
    axis=1,
)

all_teams = matches["home"].unique()
team_stats = {t: {"wins": 0, "total": 0, "gf": 0, "ga": 0} for t in all_teams}

rows = []
for _, r in matches.iterrows():
    h, a = r["home"], r["away"]
    hs = team_stats.setdefault(h, {"wins": 0, "total": 0, "gf": 0, "ga": 0})
    aws = team_stats.setdefault(a, {"wins": 0, "total": 0, "gf": 0, "ga": 0})

    is_knockout = 1 if r["round"] != "Group Stage" else 0

    rows.append(
        {
            "home": h,
            "away": a,
            "is_knockout": is_knockout,
            "home_win_rate": hs["wins"] / max(hs["total"], 1),
            "away_win_rate": aws["wins"] / max(aws["total"], 1),
            "home_avg_gf": hs["gf"] / max(hs["total"], 1),
            "away_avg_gf": aws["gf"] / max(aws["total"], 1),
            "home_avg_ga": hs["ga"] / max(hs["total"], 1),
            "away_avg_ga": aws["ga"] / max(aws["total"], 1),
            "label": r["label"],
        }
    )

    hs["total"] += 1
    aws["total"] += 1
    hs["gf"] += r["home_score"]
    hs["ga"] += r["away_score"]
    aws["gf"] += r["away_score"]
    aws["ga"] += r["home_score"]
    if r["label"] == "home_win":
        hs["wins"] += 1
    elif r["label"] == "away_win":
        aws["wins"] += 1

# Enriquecer con confederación y ranking del WC 2026
team_conf = {r["name"]: r["confederation"] for _, r in teams_2026.iterrows()}
team_rank = {r["name"]: r["fifa_ranking"] or 50 for _, r in teams_2026.iterrows()}

for row in rows:
    row["home_conf"] = team_conf.get(row["home"], "UEFA")
    row["away_conf"] = team_conf.get(row["away"], "UEFA")
    row["rank_diff"] = team_rank.get(row["home"], 50) - team_rank.get(row["away"], 50)

df = pd.DataFrame(rows)
print(f"Dataset de entrenamiento: {df.shape[0]} filas, {df.shape[1]} columnas")
df.head(10)

In [ ]:
# Correlación entre features numéricos
numeric_cols = [
    "is_knockout",
    "home_win_rate",
    "away_win_rate",
    "home_avg_gf",
    "away_avg_gf",
    "home_avg_ga",
    "away_avg_ga",
    "rank_diff",
]

fig = px.imshow(
    df[numeric_cols].corr(),
    text_auto=".2f",
    title="Matriz de correlación entre features",
    color_continuous_scale="RdBu_r",
    aspect="auto",
)
fig.update_layout(height=500, template="plotly_white")
fig.show()

In [ ]:
# Distribución de features clave por resultado
fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=[
        "Win rate del local",
        "Win rate del visitante",
        "Diferencia de ranking",
    ],
)

for i, (feat, label) in enumerate(
    [
        ("home_win_rate", "Win rate local"),
        ("away_win_rate", "Win rate visitante"),
        ("rank_diff", "Ranking diff"),
    ]
):
    for result, color in [
        ("home_win", "#636EFA"),
        ("away_win", "#EF553B"),
        ("draw", "#00CC96"),
    ]:
        subset = df[df["label"] == result]
        fig.add_trace(
            go.Histogram(
                x=subset[feat],
                name=f"{result} ({label})",
                opacity=0.6,
                marker_color=color,
                showlegend=(i == 0),
            ),
            row=1,
            col=i + 1,
        )

fig.update_layout(height=350, barmode="overlay", template="plotly_white")
fig.show()

---
## 4. Entrenamiento del Modelo

Entrenamos un **Random Forest Classifier** con los siguientes parámetros:
- 100 árboles de decisión
- Semilla aleatoria fija (`random_state=42`) para reproducibilidad
- División 80/20 train/test
- Validación cruzada con 5 folds

In [ ]:
# Codificar confederaciones
le_home = LabelEncoder()
le_away = LabelEncoder()
df["home_conf_enc"] = le_home.fit_transform(df["home_conf"])
df["away_conf_enc"] = le_away.fit_transform(df["away_conf"])

features = [
    "is_knockout",
    "home_win_rate",
    "away_win_rate",
    "home_avg_gf",
    "away_avg_gf",
    "home_avg_ga",
    "away_avg_ga",
    "home_conf_enc",
    "away_conf_enc",
    "rank_diff",
]

X = df[features].fillna(0)
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train: {X_train.shape[0]} muestras")
print(f"Test:  {X_test.shape[0]} muestras")
print(f"Features: {features}")
print(f"Clases: {y.unique().tolist()}")

In [ ]:
# Entrenar Random Forest
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

accuracy = model.score(X_test, y_test)
print(f"Precisión en test set: {accuracy:.1%}")

# Validación cruzada
cv_scores = cross_val_score(model, X, y, cv=5, scoring="accuracy")
print(f"Validación cruzada (5-fold): {cv_scores.mean():.1%} ± {cv_scores.std():.1%}")

In [ ]:
# Matriz de confusión
y_pred = model.predict(X_test)
cm = confusion_matrix(y_test, y_pred, labels=model.classes_)

fig = px.imshow(
    cm,
    x=model.classes_,
    y=model.classes_,
    text_auto=True,
    title="Matriz de confusión",
    color_continuous_scale="Blues",
    aspect="auto",
)
fig.update_layout(xaxis_title="Predicción", yaxis_title="Real", height=400)
fig.show()

print("\nReporte de clasificación:")
print(classification_report(y_test, y_pred))

In [ ]:
# Importancia de features
importance = pd.DataFrame(
    {"feature": features, "importance": model.feature_importances_}
).sort_values("importance", ascending=True)

fig = px.bar(
    importance,
    x="importance",
    y="feature",
    orientation="h",
    title="Importancia de features en el modelo Random Forest",
    color="importance",
    color_continuous_scale="Viridis",
)
fig.update_layout(height=450, template="plotly_white")
fig.show()

---
## 5. Predicciones de Partidos

Ahora utilicemos el modelo entrenado para predecir resultados de partidos hipotéticos en el Mundial 2026.

In [ ]:
def predict_match(home_team, away_team):
    """Predice el resultado de un partido entre dos selecciones."""
    conn = sqlite3.connect(str(HIST_DB))
    hist = pd.read_sql("SELECT * FROM matches", conn)
    conn.close()
    hist = hist.rename(columns={"home_team": "home", "away_team": "away"})

    def _stats(name):
        wins, total, gf, ga = 0, 0, 0, 0
        for _, r in hist.iterrows():
            if r["home"] == name or r["away"] == name:
                total += 1
                is_home = r["home"] == name
                gf += r["home_score"] if is_home else r["away_score"]
                ga += r["away_score"] if is_home else r["home_score"]
                if (is_home and r["home_score"] > r["away_score"]) or (
                    not is_home and r["away_score"] > r["home_score"]
                ):
                    wins += 1
        return {
            "wins": wins,
            "total": total,
            "gf": gf / max(total, 1),
            "ga": ga / max(total, 1),
            "win_rate": wins / max(total, 1),
        }

    hs = _stats(home_team)
    aws = _stats(away_team)

    home_conf = team_conf.get(home_team, "UEFA")
    away_conf = team_conf.get(away_team, "UEFA")
    try:
        hc_enc = le_home.transform([home_conf])[0]
    except ValueError:
        hc_enc = 0
    try:
        ac_enc = le_away.transform([away_conf])[0]
    except ValueError:
        ac_enc = 0

    X_pred = pd.DataFrame(
        [
            {
                "is_knockout": 0,
                "home_win_rate": hs["win_rate"],
                "away_win_rate": aws["win_rate"],
                "home_avg_gf": hs["gf"],
                "away_avg_gf": aws["gf"],
                "home_avg_ga": hs["ga"],
                "away_avg_ga": aws["ga"],
                "home_conf_enc": hc_enc,
                "away_conf_enc": ac_enc,
                "rank_diff": team_rank.get(home_team, 50)
                - team_rank.get(away_team, 50),
            }
        ]
    )

    proba = model.predict_proba(X_pred)[0]
    classes = model.classes_.tolist()

    probs = {c: round(float(p) * 100, 1) for c, p in zip(classes, proba)}
    return {
        "home_team": home_team,
        "away_team": away_team,
        "probabilities": probs,
        "home_ranking": team_rank.get(home_team, "N/A"),
        "away_ranking": team_rank.get(away_team, "N/A"),
    }

In [ ]:
# Partidos a predecir
partidos = [
    ("Argentina", "Brazil"),
    ("France", "Germany"),
    ("Spain", "England"),
    ("Argentina", "France"),
    ("Brazil", "England"),
    ("Germany", "Spain"),
]

predicciones = [predict_match(h, a) for h, a in partidos]

# Mostrar resultados
for pred in predicciones:
    h, a = pred["home_team"], pred["away_team"]
    p = pred["probabilities"]
    print(f"\n{'=' * 50}")
    print(f"{h} ({pred['home_ranking']}) vs {a} ({pred['away_ranking']})")
    print(f"  Victoria {h}: {p.get('home_win', 0)}%")
    print(f"  Empate:      {p.get('draw', 0)}%")
    print(f"  Victoria {a}: {p.get('away_win', 0)}%")

In [ ]:
# Visualización de predicciones
fig = make_subplots(
    rows=2,
    cols=3,
    subplot_titles=[f"{p['home_team']} vs {p['away_team']}" for p in predicciones],
)

colors = {"home_win": "#636EFA", "draw": "#FECB52", "away_win": "#EF553B"}
labels = {
    "home_win": "Victoria local",
    "draw": "Empate",
    "away_win": "Victoria visitante",
}

for i, pred in enumerate(predicciones):
    row = i // 3 + 1
    col = i % 3 + 1
    probs = pred["probabilities"]
    fig.add_trace(
        go.Bar(
            x=[labels[k] for k in ["home_win", "draw", "away_win"]],
            y=[probs.get(k, 0) for k in ["home_win", "draw", "away_win"]],
            marker_color=[colors[k] for k in ["home_win", "draw", "away_win"]],
            text=[f"{probs.get(k, 0)}%" for k in ["home_win", "draw", "away_win"]],
            textposition="outside",
            showlegend=False,
        ),
        row=row,
        col=col,
    )

fig.update_layout(
    height=600, template="plotly_white", title_text="Predicciones del Mundial 2026"
)
fig.update_yaxes(range=[0, 100])
fig.show()

In [ ]:
# Heatmap de probabilidades de victoria local
equipos_heatmap = ["Argentina", "Brazil", "France", "Germany", "Spain", "England"]
heatmap_data = []

for h in equipos_heatmap:
    row = []
    for a in equipos_heatmap:
        if h == a:
            row.append(50.0)
        else:
            pred = predict_match(h, a)
            row.append(pred["probabilities"].get("home_win", 0))
    heatmap_data.append(row)

fig = px.imshow(
    heatmap_data,
    x=equipos_heatmap,
    y=equipos_heatmap,
    text_auto=".1f",
    title="Probabilidad de victoria del equipo local (fila) vs visitante (columna)",
    color_continuous_scale="RdYlGn",
    labels=dict(color="% victoria"),
)
fig.update_layout(height=500, template="plotly_white")
fig.show()

---
## 6. Vista Previa del Dashboard

El análisis presentado aquí se complementa con un **dashboard interactivo** construido con Dash:

### Características del dashboard
- 7 pestañas: Resumen, Goles, Estadios, Equipos, Eliminatorias, Predicciones ML, Comparar Equipos
- Selector de equipos para predicciones en tiempo real
- Radar chart para comparar hasta 4 selecciones
- Gráficos interactivos con Plotly

### Ejecutar el dashboard

```bash
cd dashboard
python app.py
```

El dashboard estará disponible en `http://localhost:8050`.

![Preview del Dashboard](../data/export/embed_snippets.md)

---
## 7. Conclusiones

### Hallazgos principales

1. **Ventaja de localía:** El modelo captura consistentemente la ventaja de jugar como local, con un promedio de ~40-45% de probabilidad de victoria local en partidos entre selecciones de similar nivel.

2. **Features más importantes:**
   - La **tasa de victorias acumulada** (`home_win_rate`, `away_win_rate`) es el predictor más fuerte del modelo.
   - La **diferencia de ranking FIFA** (`rank_diff`) también tiene un peso significativo, validando que el ranking refleja el rendimiento real de las selecciones.
   - El **promedio de goles a favor** (`home_avg_gf`, `away_avg_gf`) aporta información complementaria sobre el poderío ofensivo.

3. **Rendimiento del modelo:**
   - El Random Forest alcanza una precisión competitiva considerando la naturaleza impredecible del fútbol.
   - La validación cruzada confirma que el modelo generaliza bien a partidos no vistos.

4. **Clásicos sudamericanos:**
   - Argentina vs Brasil muestra probabilidades relativamente equilibradas, reflejando la historicidad de ambos equipos en Mundiales.
   - Los equipos europeos con mejor ranking muestran mayor consistencia en sus predicciones.

### Limitaciones

- El dataset de entrenamiento contiene solo partidos de eliminatorias (48 partidos en 3 mundiales), lo cual limita la robustez estadística.
- No se consideran factores como lesiones, forma reciente, o condiciones del día del partido.
- El modelo asume independencia entre partidos, cuando en realidad el rendimiento puede variar según la fase del torneo.

### Posibles mejoras

- Incorporar datos de eliminatorias continentales (Copa América, Eurocopa, etc.)
- Añadir features de forma reciente (últimos 10 partidos)
- Probar modelos más avanzados (XGBoost, redes neuronales)
- Incluir datos de jugadores individuales y lesiones